# Notebook 07 — Stage 3: carbonate-system integrity checks

**Role in the pipeline:** the first stage with *scientific* QC, not
just data-engineering QC. Reads Stage 2's `enhanced.csv` and runs
row-wise **internal-consistency checks** of the carbonate chemistry:
does `DIC` balance with the sum of its three species (CO₂(aq), HCO₃⁻,
CO₃²⁻)? Does observed `ph_best` agree with calculated `ph_co2sys`
within the expected tolerance? Are the pH scales and units consistent?

```text
06_stage2.ipynb
   └── <stage2_out>/data/enhanced.csv
                             │
                             ▼
                  07_stage3.ipynb           ◄── THIS NOTEBOOK
                     • re-resolve canonical aliases defensively
                     • add helper columns such as sample_month, depth_round_m, lat, lon
                     • run DIC species-sum checks
                     • run pH best-vs-CO2SYS diagnostics
                     • carry solver and input-pair provenance separately
                     • write enhanced.csv, mismatch tables, QC summary, and report
                                                  │
                                                  ▼
                                          08_stage4.ipynb
```

The carbonate-system relation used here is:

```text
DIC = CO₂(aq) + HCO₃⁻ + CO₃²⁻
```

**What Stage 3 does not do:** it does not rebuild `ph_best`,
`ph_co2sys`, `ta_best_umolkg`, or other core chemistry fields. The
integrity checks surface inconsistencies and provenance gaps; the analyst
or Stage 4 audit layer decides how to classify them. Flags are advisory,
not destructive.


## Parameters

Single tagged `parameters` cell. Default `INPUT_CSV` points at the
new short Stage 2 path (`<stage2_out>/data/enhanced.csv`), not the
original's `oa_stage2_outputs\stage2_enhanced.csv` (which the
refactored pipeline never produces).


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================

# --- I/O -------------------------------------------------------------
# Set by run_pipeline.sh during normal pipeline execution.
INPUT_CSV = None
OUT_DIR = None

# --- Config override (optional) ------------------------------------
# Deep-merges onto roap.stage3.STAGE3_DEFAULTS. Use to override
# thresholds, aliases, accepted_ph_scales, or qc_group_keys without editing code.
CONFIG_PATH = None

# --- Stage 3 behaviour ---------------------------------------------
DEPTH_ROUND_DECIMALS = 1
NO_PARQUET = False
DRY_RUN = False


## Setup

All Stage 3 logic lives in `roap.stage3`: `run(df, stage_config)` re-resolves
canonical aliases, adds helper columns, runs the carbonate-system integrity
checks and the QC roll-up, and returns a `StageResult` without touching
disk. This notebook reads the input, calls `run`, shows the tables and
writes the result.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None

from roap import stage3
from roap.common import as_bool, die, is_none_like
from roap.stage_config import load_stage_config
from roap.stage_result import write_stage_result

## Load input and canonicalise

The same defensive alias resolution as Stage 2: no-op when fed Stage 2's
output, recovers if fed some other CSV that uses different names.


In [ ]:
NO_PARQUET = as_bool(NO_PARQUET)
DRY_RUN = as_bool(DRY_RUN)

try:
    DEPTH_ROUND_DECIMALS = int(DEPTH_ROUND_DECIMALS)
except Exception:
    die(f"DEPTH_ROUND_DECIMALS must be an integer, got {DEPTH_ROUND_DECIMALS!r}")

if is_none_like(INPUT_CSV):
    die("INPUT_CSV is required. Run through run_pipeline.sh or set INPUT_CSV.")
if is_none_like(OUT_DIR):
    die("OUT_DIR is required. Run through run_pipeline.sh or set OUT_DIR.")
config_path = None if is_none_like(CONFIG_PATH) else str(CONFIG_PATH).strip()

input_csv = Path(str(INPUT_CSV).strip()).expanduser().resolve()
if not input_csv.exists():
    die(f"File not found: {input_csv}\nDid Stage 2 run successfully? Stage 3 reads its enhanced.csv.")
if input_csv.suffix.lower() not in {".csv", ".txt"}:
    die(f"Expected a CSV-like file, got: {input_csv.name}")
out_root = Path(str(OUT_DIR).strip()).expanduser().resolve()

# Stage 3 defaults <- this Stage's optional config file, and nothing else (#41).
stage_cfg = load_stage_config("stage3", config_path)
for _warning in stage_cfg.warnings:
    print(f"WARNING: {_warning}")

try:
    result = stage3.run(
        pd.read_csv(input_csv), stage_cfg, source=str(input_csv), depth_round_decimals=DEPTH_ROUND_DECIMALS
    )
except ValueError as exc:
    die(str(exc))

result.manifest["output_root"] = str(out_root)
result.manifest["parameters"].update(
    {
        "INPUT_CSV": str(input_csv),
        "OUT_DIR": str(out_root),
        "CONFIG_PATH": CONFIG_PATH,
        "config_path_resolved": config_path,
        "NO_PARQUET": NO_PARQUET,
        "DRY_RUN": DRY_RUN,
    }
)
df = result.data["data/enhanced.csv"]
tables = result.tables
summary = result.manifest["integrity_summary"]

print(f"Rows loaded: {len(df):,}")
print(f"Columns    : {df.shape[1]}")
print(f"Config path: {config_path or '(defaults)'}")
if display is not None:
    display(tables["tables/canonical_presence.csv"])

## Add helper columns and column inventory

Helper columns added by `add_canonical_helper_columns`:

- `sample_month` (derived from `sample_date`)
- `depth_round_m` (from `depth_m`, rounded to `DEPTH_ROUND_DECIMALS`)
- `lat`, `lon` (aliases of `latitude_deg`, `longitude_deg`)
- normalised `ph_scale_*_normalized` and `*_unit_normalized` columns
- roll-up flags: `flag_stage2_replicate_conflict_carried`,
  `flag_solver_unknown`, `flag_carbon_input_pair_unknown`


In [ ]:
if display is not None:
    display(tables["tables/column_inventory.csv"].head(20))

## Carbonate-system integrity checks

Three families of tests are defined in `roap.stage3.carbonate_integrity_checks`:

1. **DIC species-sum check.** Flags rows where
   `|DIC - (CO2aq + HCO3 + CO3)|` exceeds
   `max(dic_abs_tol, |DIC|*dic_rel_tol)`. The robust variant uses
   MAD outlier detection on the residual distribution instead of
   only a fixed threshold.
2. **pH diagnostic.** Flags rows where `|ph_best - ph_co2sys|`
   exceeds `ph_diag_tol`. The strict variant requires known matching
   pH scales. The robust variant uses MAD on the residual.
3. **Scale + unit consistency.** Per-row flags for missing scale
   context, unexpected pH scales, mismatched pH scales between observed
   and calculated pH, and mismatched units across the DIC species.

A roll-up `flag_any_carbonate_issue` is `True` if any non-strict
chemistry issue fires. A stricter `flag_any_carbonate_issue_strict`
includes severe carbonate consistency problems such as DIC inconsistency,
pH scale mismatch, unexpected pH scale, and scale-aware pH diagnostic
mismatch.

Solver and carbon input-pair provenance are carried separately for
Stage 4 audit decisions. They are not folded into the strict Stage 3
chemistry roll-up.


In [ ]:
print(f"Any carbonate issue : {summary['n_any_carbonate_issue']:,} rows")
print(f"                     (strict: {summary['n_any_carbonate_issue_strict']:,})")
print(f"DIC checkable        : {summary['n_dic_checkable']:,}")
print(f"DIC inconsistent     : {summary['n_dic_inconsistent']:,}")
print(f"pH checkable         : {summary['n_ph_checkable']:,}")
print(f"pH diagnostic mismatch: {summary['n_ph_diag_mismatch']:,}")
print(f"Accepted pH scales   : {result.manifest['accepted_ph_scales']}")
if display is not None:
    display(pd.DataFrame([summary]))
    for title, rel in (("DIC mismatches", "tables/dic_species_mismatches.csv"), ("pH mismatches", "tables/ph_diag_mismatches.csv")):
        if not tables[rel].empty:
            print(f"\n{title}:")
            display(tables[rel].head(20))

## QC summary by group

Aggregates the flag counts to `[cruise_id, transect_id, station_id,
depth_round_m, sample_month]` (by default; configurable via
`qc_group_keys`). Useful for spotting whether the failures cluster
in a particular cruise / station / season.

The summary is also joined back to the row-level frame so each row
carries its group's roll-up counts — handy for downstream filtering.


In [ ]:
qc_df = tables["tables/qc_summary_by_group.csv"]
print(f"QC groups: {len(qc_df):,} | group keys used: {result.manifest['qc_group_keys_used']}")
if display is not None:
    display(qc_df.head(20))

## Quick preview

In [ ]:
preview_cols = [
    c for c in [
        "record_id", "sample_id", "sample_date", "sample_month",
        "station_id", "depth_m", "depth_round_m", "lat", "lon",
        "ta_best_umolkg", "ph_best", "ph_co2sys",
        "dic_best_umol_kg", "co2aq_calc_umol_kg",
        "hco3_calc_umol_kg", "co3_calc_umol_kg",
        "flag_dic_inconsistent", "flag_ph_diag_mismatch",
        "flag_any_carbonate_issue", "flag_any_carbonate_issue_strict",
    ]
    if c in df.columns
]
if display is not None:
    display(df[preview_cols].head(20))
else:
    print(df[preview_cols].head(20).to_string(index=False))


## Write outputs

```
<OUT_DIR>/
    data/enhanced.csv   (and .parquet)   ◄── Stage 4 input
    tables/carbonate_integrity_flags.csv, qc_summary_by_group.csv,
           dic_species_mismatches.csv, ph_diag_mismatches.csv,
           column_inventory.csv, canonical_presence.csv, alias_resolution.csv
    reports/report.md
    logs/manifest.json, effective_config.json
    run_bundle_manifest.json
```

`DRY_RUN = True` skips the writes. `NO_PARQUET = True` writes CSV only.
The run bundle hashes the input, the config and every file this run
wrote, so `roap verify-bundle <OUT_DIR>` can check them later.

In [ ]:
if DRY_RUN:
    print("DRY_RUN = True -- no files written.")
else:
    written = write_stage_result(result, out_root, parquet=not NO_PARQUET)
    print(f"Stage 3 complete: {len(written)} files under {out_root}")
    print(f"  -> Stage 4 input: {written['data/enhanced.csv']}")